### Note - This file contains code for merging TL models with Base models
### However, there are 2 types of merging - 1. Existing base Models and 2. Runtime/Direct Base Models
### Code needs to be commented and / or uncommented based on the type of merging
### Currently code is as per method 2

# 1. Importing Libraries and Setting Paths


In [1]:
# import logging

# logging.disable(logging.WARNING)

import itertools
import os

# import joblib
import numpy as np
import pandas as pd

import sklearn
from sklearn import tree
import tensorflow
from sklearn.gaussian_process.kernels import (
    RBF,
    ConstantKernel,
    DotProduct,
    WhiteKernel,
)
from sklearn.model_selection import GridSearchCV
from sklearn.svm import SVC
from tensorflow.keras.models import Model, load_model  # , save_model

# import warnings
pd.set_option("display.max_columns", None)

os.environ["LOKY_MAX_CPU_COUNT"] = "4"  # Set to the desired number of cores

from TL_BaseModel import *

In [2]:
# Error will occur if pandas greater than specified due to loss of backward compatibility
# https://stackoverflow.com/questions/75953279/modulenotfounderror-no-module-named-pandas-core-indexes-numeric-using-metaflo
# pip install "pandas<2.0.0"
print("pd.__version__ = ", pd.__version__)  # 1.4.0 should be <2
# print("keras.__version__ = ", keras.__version__)  # 2.15.0
print("tensorflow.__version__ = ", tensorflow.__version__)  # 2.15.0
print(
    "sklearn.__version__ = ", sklearn.__version__
)  # Should be 1.2.2 because classification_results_2023 were generated with scikit-learn 1.2.2

pd.__version__ =  1.5.3
tensorflow.__version__ =  2.15.0
sklearn.__version__ =  1.4.0


In [3]:
# Below mentioned paths are updated below for testing purpose to verify results after removal of kfolds
results_path = "./../../../Output_Results/"

tl_results_path = results_path + "SDDSmoteBoost_Unf_All_Patient_Direct_Base"
# base_results_path = results_path + "classification_results_2023"

print("results_path", results_path)
print("tl_results_path", tl_results_path)
# print("base_results_path", base_results_path)

results_path ./../../../Output_Results/
tl_results_path ./../../../Output_Results/SDDSmoteBoost_Unf_All_Patient_Direct_Base


# 2. Settings - To Control Execution


In [4]:
# Possible Values: "TL_1_Frozen", "TL_2_Frozen", "TL_3_Frozen", "TL_All_Frozen", "TL_All_Unfrozen"
Architecture_GROUP_List = [
    "TL_All_Unfrozen",
]

Model_Type_list = [
    "K-NN",
    "LogisticRegression",
    "NaiveBayes",
    "SVM",
    "DecisionTree",
]

EVALUATION_METRICS = [
    "Weighted Accuracy",
    "Sensitivity/Recall",
    "Specificity",
    "Precision_class0",
    "Precision_class1",
    "Precision_avg",
    "F1_class0",
    "F1_class1",
    "F1_avg",
    "auc_roc_score",
    "False_Discovery_Rate",
    "False_Negative_Rate",
    "False_Omission_Rate",
    "False_Positive_Rate",
    "Jaccard",
]

# Number of K splits for k-fold cross validation for each model in the list
number_of_splits = 3

# CHANGE ACCORDING TO YOUR ALGO
# Possible Values: ADKNN , GammaSmoteBoost,  <<-- Other Teams append their Algorithms here -->>
augmentation_algo = "GammaSmoteBoost"
augmentation_algo = "ADKNN"
augmentation_algo = "SMOTEBoostCC"
augmentation_algo = "SDDSMOTEBoostCC"

# WHEN FALSE, TARGET DATA IS NOT REMOVED FROM ALL PATIENTS DATA
# False for Generalized model, True for Patient-wise Pop Models
bool_remove_target = False

# Number of Folds for running Pop Model
# Possible values : [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
num_pop_folds = 5

# When True will drop Age, BMI and Sex columns too else will keep them too
bool_drop_ABS = True

# Number of Folds for running Ind Model
# Possible values : [3, 4, 5]
num_ind_folds = 3

POPULATION_EPOCH_SIZE_LIST = [100, 300, 500, 700, 1000, 2000, 3000]

PATIENT_EPOCH_SIZE_LIST = [100, 300, 500, 700, 1000, 2000, 3000]

target_grp = [
    "SB-001",
    "SB-003",
    "SB-008",
    "SB-011",
    "SB-012",
    "SB-022",
    "SB-023",
    "SB-025",
    "SB-028",
    "SB-037",
    "SB-043",
    "SB-056",
    "SB-059",
    "SB-060",
    "SB-070",
    "SB-071",
    "SB-073",
    "SB-078",
    "SB-079",
    "SB-080",
    "SB-082",
    "SB-083",
    "SB-112",
]

# 3. finding all TL models


In [6]:
# Get all TL models
# Keras=2.15.0 and Using tensorflow Load Model
def find_and_load_all_TL_models(tl_results_path, Architecture_GROUP_List, target_grp):
    TL_Models_list = {}
    loaded_Model_dict = {}
    not_loaded_Model_dict = {}

    for target_id in target_grp:
        Model_List = {}
        loaded_Model_dict_inner = {}
        not_loaded_Model_dict_inner = {}

        for Architecture_GROUP in Architecture_GROUP_List:
            Model_path = f"{tl_results_path}/{Architecture_GROUP}/diagrams_n_individual_models/{target_id}/"
            if os.path.exists(Model_path) and os.path.isdir(Model_path):
                for file_name in os.listdir(Model_path):
                    if file_name.endswith("keras"):
                        model_filepath = os.path.join(Model_path, file_name)
                        try:
                            model = load_model(model_filepath)
                            Model_List[file_name] = model_filepath
                            loaded_Model_dict_inner[file_name] = model_filepath
                        except Exception as e:
                            not_loaded_Model_dict_inner[file_name] = model_filepath

        TL_Models_list[target_id] = Model_List
        loaded_Model_dict[target_id] = loaded_Model_dict_inner
        not_loaded_Model_dict[target_id] = not_loaded_Model_dict_inner

        print(f"{target_id} completed")

    return TL_Models_list, loaded_Model_dict, not_loaded_Model_dict


TL_Models_list, loaded_Model_dict, not_loaded_Model_dict = find_and_load_all_TL_models(
    tl_results_path, Architecture_GROUP_List, target_grp
)

Total number of Models in SB-001 : 15
Total number of Models in SB-003 : 15
Total number of Models in SB-008 : 15
Total number of Models in SB-011 : 15
Total number of Models in SB-012 : 15
Total number of Models in SB-022 : 15
Total number of Models in SB-023 : 15
Total number of Models in SB-025 : 15
Total number of Models in SB-028 : 15
Total number of Models in SB-037 : 15
Total number of Models in SB-043 : 15
Total number of Models in SB-056 : 15
Total number of Models in SB-059 : 15
Total number of Models in SB-060 : 15
Total number of Models in SB-070 : 15
Total number of Models in SB-071 : 15
Total number of Models in SB-073 : 15
Total number of Models in SB-078 : 15
Total number of Models in SB-079 : 15
Total number of Models in SB-080 : 15
Total number of Models in SB-082 : 15
Total number of Models in SB-083 : 15
Total number of Models in SB-112 : 15


# 4. Data Loading

In [12]:
# GLOBALS AND FILEPATHS#
source_quantile = "0.2"
source_file_path = f"{execution_path}Input_Data/Og_Data/labeled_normal_data/all_pids_q={source_quantile}.npy"
display(source_file_path)

##### KEY TROUBLESHOOTING NOTE
##### np.load returns a numpy ndarray

data_ndarr = np.load(source_file_path, allow_pickle=True)
print("data_ndarr type :", type(data_ndarr))

##### That array has a method item(),
##### which returns a dictionary of key-values: p_id:DataFrame
data_dict = data_ndarr.item()
print("data_dict type :", type(data_dict))

p_ids = data_dict.keys()
sample_size_dict = {}
print("patients: n =", len(p_ids), end="/n/n")

for p_id in p_ids:
    df = data_dict[p_id]
    # drop column from input data file
    df.drop(["id", "date", "am_pef_org", "BMI", "sex", "age"], axis=1, inplace=True)
    df = np.asarray(df).astype(np.float32)
    print(p_id, "shape:", df.shape)  # , end="/t/t")
    sample_size_dict[p_id] = df.shape[0]

display(data_dict[p_id].head(5))
display(sample_size_dict)

'../../../Input_Data/Og_Data/labeled_normal_data/all_pids_q=0.2.npy'

data_ndarr type : <class 'numpy.ndarray'>
data_dict type : <class 'dict'>
patients: n = 26/n/nSB-001 shape: (188, 28)
SB-003 shape: (180, 28)
SB-008 shape: (178, 28)
SB-011 shape: (210, 28)
SB-012 shape: (194, 28)
SB-022 shape: (204, 28)
SB-023 shape: (194, 28)
SB-025 shape: (117, 28)
SB-028 shape: (92, 28)
SB-033 shape: (53, 28)
SB-037 shape: (210, 28)
SB-043 shape: (191, 28)
SB-056 shape: (191, 28)
SB-059 shape: (203, 28)
SB-060 shape: (88, 28)
SB-070 shape: (202, 28)
SB-071 shape: (124, 28)
SB-073 shape: (161, 28)
SB-078 shape: (153, 28)
SB-079 shape: (142, 28)
SB-080 shape: (181, 28)
SB-081 shape: (34, 28)
SB-082 shape: (137, 28)
SB-083 shape: (206, 28)
SB-089 shape: (49, 28)
SB-112 shape: (180, 28)


,income,road_dist,cooking,y_am_pef,tempin,humidin,pm25in,co2in,tempdiffin,humidiffin,pm25diffin,pm10,pm25,o3,no2,co,so2,temp,windsd,humid,varp,dewpt,airp,seap,solrhr,solramnt,grdt,class
1,0.857,0.0,0.167,0.558,0.216,0.513,0.358,0.678,0.562,0.364,0.619,0.390,0.250,0.024,0.587,0.536,0.222,0.349,0.107,0.668,0.174,0.471,0.822,0.819,0.165,0.003,0.206,0.0
2,0.857,0.0,0.167,0.006,0.299,0.521,0.411,0.700,0.436,0.672,0.616,0.446,0.285,0.000,0.783,0.732,0.222,0.473,0.171,0.388,0.190,0.491,0.676,0.670,0.603,0.165,0.383,0.0
3,0.857,0.0,0.167,0.078,0.420,0.713,0.372,0.918,0.469,0.821,0.734,0.466,0.155,0.190,0.239,0.262,0.111,0.474,0.515,0.359,0.197,0.481,0.519,0.518,0.485,0.104,0.346,1.0
4,0.857,0.0,0.167,0.428,0.438,0.468,0.224,0.762,0.560,0.490,0.794,0.211,0.011,0.143,0.196,0.110,0.111,0.404,0.338,0.527,0.175,0.467,0.596,0.596,0.491,0.141,0.339,0.0
5,0.857,0.0,0.167,0.209,0.462,0.385,0.199,0.502,0.612,0.529,0.725,0.250,0.057,0.095,0.348,0.245,0.111,0.366,0.316,0.444,0.141,0.419,0.765,0.762,0.433,0.135,0.281,1.0


{'SB-001': 188,
 'SB-003': 180,
 'SB-008': 178,
 'SB-011': 210,
 'SB-012': 194,
 'SB-022': 204,
 'SB-023': 194,
 'SB-025': 117,
 'SB-028': 92,
 'SB-033': 53,
 'SB-037': 210,
 'SB-043': 191,
 'SB-056': 191,
 'SB-059': 203,
 'SB-060': 88,
 'SB-070': 202,
 'SB-071': 124,
 'SB-073': 161,
 'SB-078': 153,
 'SB-079': 142,
 'SB-080': 181,
 'SB-081': 34,
 'SB-082': 137,
 'SB-083': 206,
 'SB-089': 49,
 'SB-112': 180}

# 5. TL_and_Base_Model function definition


In [14]:
def TL_and_Base_Model(
    target_grp,
    tl_results_path,
    TL_Models_list,
    # Base_Models_list,
    data_dict,
    EVALUATION_METRICS,
    Architecture_GROUP_List,
    Model_Type_list,
    number_of_splits,
    augmentation_algo,
):
    for target_id in target_grp:
        df_evaluation_results = pd.DataFrame(columns=EVALUATION_METRICS)
        # Mark: EXPLICITLY set binary outcome variable for supervised learning
        CLASS_VAR = "class"

        # load target patient data
        dataset_list = split_data(
            target=target_id,
            data=data_dict,
            class_var=CLASS_VAR,
            balance=True,
            test_frac=0.2,
            transfer=False,
            kfold_n_splits=number_of_splits,
            augmentation_algo=augmentation_algo,
        )

        TL_Models = TL_Models_list[target_id]
        # Base_Models = Base_Models_list[target_id]

        for Model_Type, Architecture_GROUP in itertools.product(
            Model_Type_list, Architecture_GROUP_List
        ):
            if os.path.exists(f"{tl_results_path}/{Architecture_GROUP}/"):
                # Creating Category directory for the patient
                os.makedirs(
                    os.path.dirname(
                        f"{tl_results_path}/{Architecture_GROUP}/diagrams_n_TL-Base_models/"
                    ),
                    exist_ok=True,
                )

                output_results_path = (
                    f"{tl_results_path}/{Architecture_GROUP}/diagrams_n_TL-Base_models/"
                )

                for TL_Model_filename, TL_Model_filepath in TL_Models.items():
                    if Architecture_GROUP in TL_Model_filename:
                        model = load_model(TL_Model_filepath, compile=False)

                        tl_model_for_base = Model(
                            inputs=model.input,
                            outputs=model.layers[-2].output,
                        )

                        print(
                            f"{target_id} - {Model_Type} - {Architecture_GROUP} - {TL_Model_filename}"
                        )

                        # for (
                        #     Base_Model_filename,
                        #     Base_Model_filepath,
                        # ) in Base_Models.items():
                        #     if Model_Type in Base_Model_filename:
                        #         print(
                        #             f"{target_id} - {Model_Type} - {Architecture_GROUP} - {Base_Model_filename} - {TL_Model_filename}"
                        #         )

                        #         Base_model = joblib.load(Base_Model_filepath)

                        # setup for k-fold cross-validation
                        kfold_evaluation_results_list = []
                        kfold_confusion_matrix_list = []

                        for i in range(number_of_splits):
                            print(
                                f"================================================{i}================================================"
                            )
                            X_train, X_test, Y_train, Y_test = dataset_list[i]

                            # Now, let us use training features from neural network for Base
                            feature_extractor = tl_model_for_base.predict(X_train)
                            X_train_for_base = feature_extractor.reshape(
                                feature_extractor.shape[0], -1
                            )  # This is our X input to Base

                            # # Train the model on training data
                            classifier_model = None

                            if Model_Type == "NaiveBayes":
                                parameters = {
                                    "kernel": [
                                        ConstantKernel(
                                            1.0, constant_value_bounds="fixed"
                                        )
                                        * RBF(1.0, length_scale_bounds="fixed"),
                                        1.0 * RBF(1.0),
                                        DotProduct() + WhiteKernel(),
                                    ]
                                }

                                nb = sklearn.gaussian_process.GaussianProcessClassifier(
                                    random_state=42
                                )

                                GridNB = GridSearchCV(
                                    estimator=nb, param_grid=parameters, cv=5
                                )
                                GridNB.fit(X_train, Y_train.ravel())
                                classifier_model = GridNB.best_estimator_  # cl

                            elif Model_Type == "DecisionTree":
                                parameters = {
                                    "max_depth": (
                                        2,
                                        3,
                                        4,
                                        5,
                                        6,
                                        7,
                                        8,
                                        9,
                                        10,
                                        11,
                                        12,
                                        13,
                                        14,
                                        15,
                                        16,
                                        17,
                                        18,
                                        19,
                                        20,
                                        21,
                                        22,
                                    ),
                                    "criterion": ("entropy", "gini"),
                                }

                                Grid_DT_depth = GridSearchCV(
                                    tree.DecisionTreeClassifier(),
                                    parameters,
                                    cv=5,
                                )
                                Grid_DT_depth.fit(X_train, Y_train.ravel())
                                classifier_model = Grid_DT_depth.best_estimator_

                            elif Model_Type == "LogisticRegression":
                                parameters = {
                                    "penalty": ["l1", "l2"],
                                    "C": [0.001, 0.01, 0.1, 1, 10, 100, 1000],
                                    "solver": ["liblinear", "lbfgs"],
                                }  # "C":np.logspace(-3,3,7)
                                GridLR = GridSearchCV(
                                    sklearn.linear_model.LogisticRegression(),
                                    param_grid=parameters,
                                    cv=5,
                                )
                                GridLR.fit(X_train, Y_train.ravel())
                                classifier_model = GridLR.best_estimator_  # cl

                            elif Model_Type == "K-NN":
                                parameters = {
                                    "n_neighbors": [3, 5, 11, 9],
                                    "weights": ["uniform", "distance"],
                                    "metric": ["euclidean", "manhattan"],
                                }

                                GridKNN = GridSearchCV(
                                    sklearn.neighbors.KNeighborsClassifier(),
                                    param_grid=parameters,
                                    cv=5,
                                )
                                GridKNN.fit(X_train, Y_train.ravel())
                                classifier_model = GridKNN.best_estimator_  # cl

                            elif Model_Type == "SVM":
                                parameters = {
                                    "kernel": ("linear", "poly", "rbf", "sigmoid"),
                                    "C": np.arange(1, 42, 10),
                                    "degree": np.arange(3, 6),
                                    "coef0": np.arange(0.001, 3, 0.5),
                                    "gamma": ("auto", "scale"),
                                }
                                SVModel = SVC()
                                GridS = GridSearchCV(SVModel, parameters, cv=5)
                                GridS.fit(X_train, Y_train.ravel())
                                classifier_model = GridS.best_estimator_  # cl

                            classifier_model.fit(X_train_for_base, Y_train.ravel())

                            # Send test data through same feature extractor process
                            X_test_feature = tl_model_for_base.predict(X_test)
                            # X_test_features = X_test_feature.reshape(
                            #     X_test_feature.shape[0], -1
                            # )

                            # Now predict using the trained Base model.
                            y_test_prediction = classifier_model.predict(X_test_feature)

                            TL_Base_results, c_matrix = evaluate(
                                y_test_prediction.reshape(
                                    y_test_prediction.shape[0], -1
                                ),
                                Y_test,
                            )

                            kfold_evaluation_results_list.append(TL_Base_results)
                            kfold_confusion_matrix_list.append(c_matrix)

                        version_tag = f"{TL_Model_filename[:-6]}_{Model_Type}"

                        avg_results = (
                            pd.DataFrame(
                                kfold_evaluation_results_list,
                                columns=EVALUATION_METRICS,
                            )
                            .mean()
                            .values
                        ).reshape(1, 15)

                        avg_results_df = pd.DataFrame(
                            avg_results, columns=EVALUATION_METRICS
                        )

                        avg_results_df.index = [version_tag]

                        df_evaluation_results = pd.concat(
                            [df_evaluation_results, avg_results_df]
                        )

                    evaluation_results_save_path = os.path.join(
                        f"{output_results_path}TL_Base_{Architecture_GROUP}_evaluation_metrics.csv"
                    )

                    if os.path.isfile(evaluation_results_save_path):
                        df_evaluation_results.to_csv(
                            evaluation_results_save_path, mode="a", header=False
                        )
                    else:
                        df_evaluation_results.to_csv(
                            evaluation_results_save_path, mode="a"
                        )
                    df_evaluation_results = pd.DataFrame(columns=EVALUATION_METRICS)

# 6. Function Execution


In [15]:
TL_and_Base_Model(
    target_grp,
    tl_results_path,
    loaded_Model_dict,
    data_dict,
    EVALUATION_METRICS,
    Architecture_GROUP_List,
    Model_Type_list,
    number_of_splits,
    augmentation_algo,
)

~~~~~~~ value_counts: ~~~~~~~
SHOULD BE BALANCED NOW:
 0.0    94
1.0    94
Name: class, dtype: int64
~~~~~~~ value_counts: ~~~~~~~
SHOULD BE BALANCED NOW:
 0.0    103
1.0    103
Name: class, dtype: int64
~~~~~~~ value_counts: ~~~~~~~
SHOULD BE BALANCED NOW:
 1.0    105
0.0    105
Name: class, dtype: int64
SB-001 - DecisionTree - TL_All_Unfrozen - source-_size__5182_TL_All_Unfrozen_100_POP_FOLD_1_SB-001-3000_IND_FOLD_1.keras
================================================0================================================
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step 
================================================1================================================
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step 
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step 
================================================2================================================
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step 
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step 
SB-001 - DecisionTree - TL_All_Unfrozen - source-_

### Formatting Evaluations to expected sequence of rows

In [5]:
from Eval_Formatter import main

main()